## This is a notebook for our churn prediction model pipeline

### Connect to Google BigQuery Database

### Look at features - EDA

### Feature engineering - Customer segmentation (Kmeans), Time series analysis

### Model selection

### Model evaulation


In [1]:
import gc
import time
import platform
import os
from dataclasses import dataclass
from typing import Callable, Any, List, Dict

import numpy as np
import pandas as pd

from google.cloud import bigquery
import db_dtypes


### Configure bigquery credentials

#### you may need to download gcloud - run 'brew install --cask google-cloud-sdk' and follow instructions to add to path

#### change the sql query to get different joined tables


In [10]:
PROJECT_ID = 'netflix-user-behavior'
# can add different queries based on what we need. 
# project id stays the same 
SQL = os.environ.get(
    "BQ_SQL",
    """
    SELECT
    m.*,
    w.*
FROM `netflix-user-behavior.kaggle_cleaned.movies_cleaned` AS m
JOIN `netflix-user-behavior.kaggle_cleaned.watch_history_cleaned` AS w
    ON m.movie_id = w.movie_id
LIMIT 1000;
    """.strip()
)
SQL_USER_BEHAVIOR = os.environ.get(
	"BQ_SQL_USER_BEHAVIOR",

	"""
	SELECT * 
	FROM `netflix-user-behavior.kaggle_cleaned.users_cleaned`
	"""
)
print("PROJECT_ID:", PROJECT_ID)
print("SQL preview:\n", SQL[:300], "..." if len(SQL) > 300 else "")
print("SQL_USER_BEHAVIOR preview:\n", SQL_USER_BEHAVIOR[:300], "..." if len(SQL_USER_BEHAVIOR) > 300 else "")

PROJECT_ID: netflix-user-behavior
SQL preview:
 SELECT
    m.*,
    w.*
FROM `netflix-user-behavior.kaggle_cleaned.movies_cleaned` AS m
JOIN `netflix-user-behavior.kaggle_cleaned.watch_history_cleaned` AS w
    ON m.movie_id = w.movie_id
LIMIT 1000; 
SQL_USER_BEHAVIOR preview:
 
	SELECT * 
	FROM `netflix-user-behavior.kaggle_cleaned.users_cleaned`
	 


In [ ]:
# load in data from bigquery to pandas dataframe.
def load_bigquery_to_pandas(project_id: str, sql: str) -> pd.DataFrame:
	# authenticates using google credentials and connects to bigquery
	client = bigquery.Client(project=project_id)
    # sends request to bigquery and returns QueryJob object
	job = client.query(sql)
	df = job.result().to_dataframe()
	return df

#
pdf = load_bigquery_to_pandas(PROJECT_ID, SQL)
user = load_bigquery_to_pandas(PROJECT_ID, SQL_USER_BEHAVIOR)

/Users/allisonpeng/miniconda3/envs/netflix-env/lib/python3.11/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
/Users/allisonpeng/miniconda3/envs/netflix-env/lib/python3.11/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [19]:
user.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6735 entries, 0 to 6734
Data columns (total 16 columns):
 #   Column                   Non-Null Count  Dtype              
---  ------                   --------------  -----              
 0   user_id                  6735 non-null   object             
 1   email                    6735 non-null   object             
 2   first_name               6735 non-null   object             
 3   last_name                6735 non-null   object             
 4   age                      6735 non-null   float64            
 5   gender                   6735 non-null   object             
 6   country                  6735 non-null   object             
 7   state_province           6735 non-null   object             
 8   city                     6735 non-null   object             
 9   subscription_plan        6735 non-null   object             
 10  subscription_start_date  6735 non-null   dbdate             
 11  is_active                6735 

### pull basic user information from the user table, how many unique users do we have and what is the time period we are looking at
